In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects =500
num_features = 5

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0    # X5 effect
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 0.5

Y_raw = X @ beta_true + noise

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 0.9382],
        [ 2.0201],
        [-1.5253],
        [ 0.5188],
        [ 0.0249],
        [ 3.0189]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

#print(scores)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([7, 7])

Attention Matrix:
 tensor([[8.2127e-02, 1.0392e-01, 1.2943e-01, 1.5312e-01, 4.5937e-02, 1.0256e-01,
         3.8291e-01],
        [2.0472e-01, 1.3229e-01, 1.7028e-01, 1.7450e-01, 1.5471e-01, 1.2761e-01,
         3.5898e-02],
        [1.5934e-01, 1.9915e-01, 2.1233e-01, 1.1994e-01, 1.4323e-01, 1.0449e-01,
         6.1517e-02],
        [1.4746e-01, 1.4372e-01, 2.3647e-01, 1.7104e-01, 1.4616e-01, 1.2264e-01,
         3.2512e-02],
        [8.7430e-02, 1.5699e-01, 1.4849e-01, 1.9263e-01, 1.4179e-01, 1.3961e-01,
         1.3307e-01],
        [2.3113e-02, 2.6725e-02, 1.8861e-02, 2.9082e-02, 2.6406e-02, 6.9374e-02,
         8.0644e-01],
        [8.4106e-05, 4.3435e-05, 3.8009e-05, 3.7334e-04, 5.8486e-05, 2.0726e-03,
         9.9733e-01]], grad_fn=<SoftmaxBackward0>)


In [8]:
# 把Y再從矩陣中拿掉

A = scores[:-1,:-1]

print("SCORE Matrix Shape(拿掉Y):", A.shape)
print("\nSCORE Matrix(拿掉Y):\n", A)

SCORE Matrix Shape(拿掉Y): torch.Size([6, 6])

SCORE Matrix(拿掉Y):
 tensor([[ 0.9885,  2.3198,  3.5614,  4.5124, -2.2982,  2.2453],
        [ 0.5967, -1.8735, -0.4452, -0.3068, -0.9876, -2.0770],
        [ 0.5027,  1.7642,  2.1266, -1.1043, -0.1005, -1.8841],
        [-0.2501, -0.3953,  2.4213,  0.5891, -0.3003, -1.2928],
        [-2.6108,  0.7005,  0.3854,  1.8577,  0.1243,  0.0365],
        [-0.7682,  0.0531, -1.9181,  0.5312, -0.0146,  5.4493]],
       grad_fn=<SliceBackward0>)


In [9]:
X_train.T @ X_train

tensor([[400.0000,  37.8796,  17.3058,  18.9547, -27.1923, -38.8521],
        [ 37.8796, 383.3899,  13.6642,  29.4697,  21.0218,   0.6505],
        [ 17.3058,  13.6642, 419.3848,  27.5260, -32.4748, -15.7170],
        [ 18.9547,  29.4697,  27.5260, 431.4060, -19.5070,  -1.3612],
        [-27.1923,  21.0218, -32.4748, -19.5070, 395.3122,  15.6646],
        [-38.8521,   0.6505, -15.7170,  -1.3612,  15.6646, 419.9208]])

In [10]:
# 放入OLS的公式解中，估計beta

#beta_attention=(torch.inverse(X_train.T @ X_train)@A) @ X_train.T @ Y_train

beta_attention = torch.linalg.solve(((X_train.T @ X_train) - 0.2*A),X_train.T @ Y_train)


In [11]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

In [12]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 0.9237],
        [ 2.0293],
        [-1.5324],
        [ 0.5189],
        [ 0.0256],
        [ 3.0155]])


In [13]:
print(
    "Attention MSE:",
    mse_attention.item()
)

Attention MSE: 0.28354713320732117


In [14]:
print(
    "OLS MSE:",
    mse_ols.item()
)

OLS MSE: 0.2846986651420593
